# Day 2 · 실습: ResNet 구조와 Residual Connection 확인

**연계 이론:** 이론 1 「깊은 CNN의 문제와 Residual Learning」 · 이론 2 「ResNet과 현대적인 CNN 아키텍처」

Residual 덧셈, Shortcut의 Shape, 사전학습 ResNet18의 실제 블록과 Feature Map을 확인합니다.

## 0. 준비

사전학습 ResNet18 가중치를 사용합니다. `resnet18-f37072fd.pth`가 weights 폴더 아래에 포함되어 있는지 확인해봅니다. 

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
import torchvision
from torchvision.models import ResNet18_Weights, resnet18

# 재현 가능한 예시 입력을 만들기 위한 시드입니다.
torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("실행 장치   :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU        :", torch.cuda.get_device_name(0))

## 1. Residual Connection의 덧셈 확인

**연계 이론:** 이론 1 · 4.2~5.2 / 이론 2 · 2.1

같은 변환 결과 `F(x)`에서 **입력 `x`를 더하기 전과 후**의 차이를 확인합니다. 여기의 `F(x)`는 계산 원리를 보여 주기 위해 정한 예시입니다.

In [ ]:
# 이해를 위한 2×2 예시: F(x)는 미리 정한 값이며, 학습된 Conv 출력이 아닙니다.
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
fx = torch.tensor([[-0.5, 0.5], [0.25, -1.0]])

plain_output = torch.relu(fx)
residual_output = torch.relu(fx + x)

print("입력 x:\n", x)
print("변환 결과 F(x):\n", fx)
print("일반 블록 ReLU(F(x)):\n", plain_output)
print("Residual Block ReLU(F(x)+x):\n", residual_output)
print("두 출력의 Shape:", tuple(plain_output.shape), tuple(residual_output.shape))

assert torch.equal(residual_output, torch.relu(fx + x))
assert not torch.equal(plain_output, residual_output)

In [ ]:
# 네 가지 값을 같은 색상 범위로 시각화해 차이를 확인합니다.
items = [("Input x", x), ("Transform F(x)", fx),
         ("Plain output", plain_output), ("Residual output", residual_output)]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.3), constrained_layout=True)
for ax, (title, t) in zip(axes, items):
    im = ax.imshow(t.numpy(), cmap="viridis", vmin=-1, vmax=4)
    ax.set_title(title)
    ax.set_xticks(range(2)); ax.set_yticks(range(2))
    for (r, c), value in np.ndenumerate(t.numpy()):
        ax.text(c, r, f"{value:.2f}", ha="center", va="center", color="white", fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.8, label="Value")
plt.show()

## 2. Identity Shortcut과 Projection Shortcut의 Shape 비교

**연계 이론:** 이론 2 · 2.2, 3.1~3.4

두 경로의 Tensor Shape이 같을 때 덧셈이 가능합니다. `STRIDE_TO_COMPARE`를 `1`과 `2`로 바꿔 Projection 출력이 `F(x)`와 일치하는지 확인합니다.

In [ ]:
# A. Identity Shortcut: 같은 Shape을 그대로 더합니다.
x_identity = torch.randn(1, 64, 56, 56)
fx_identity = torch.randn(1, 64, 56, 56)  # 변환 경로의 출력이라고 가정
identity_shortcut = nn.Identity()

skip_identity = identity_shortcut(x_identity)
out_identity = torch.relu(fx_identity + skip_identity)
print("[Identity Shortcut]")
print("입력 x        :", tuple(x_identity.shape))
print("변환 결과 F(x):", tuple(fx_identity.shape))
print("Shortcut     :", tuple(skip_identity.shape))
print("최종 출력     :", tuple(out_identity.shape))
assert x_identity.shape == fx_identity.shape == out_identity.shape

# B. Projection Shortcut: 채널 수와 공간 크기를 변경합니다.
x_projection = torch.randn(1, 64, 56, 56)
fx_projection = torch.randn(1, 128, 28, 28)  # 다음 Stage의 변환 결과라고 가정
projection_shortcut = nn.Sequential(
    nn.Conv2d(64, 128, kernel_size=1, stride=2, bias=False),
    nn.BatchNorm2d(128),
).eval()
with torch.no_grad():
    skip_projection = projection_shortcut(x_projection)
    out_projection = torch.relu(fx_projection + skip_projection)

print("\n[Projection Shortcut]")
print("입력 x        :", tuple(x_projection.shape))
print("변환 결과 F(x):", tuple(fx_projection.shape))
print("Shortcut     :", tuple(skip_projection.shape))
print("최종 출력     :", tuple(out_projection.shape))
assert skip_projection.shape == fx_projection.shape == out_projection.shape

In [ ]:
STRIDE_TO_COMPARE = 1  # 🔧 1 또는 2로 변경해 보세요.
comparison_proj = nn.Conv2d(64, 128, kernel_size=1, stride=STRIDE_TO_COMPARE, bias=False)
with torch.no_grad():
    candidate = comparison_proj(x_projection)
print("Projection Stride:", STRIDE_TO_COMPARE)
print("Shortcut 출력  :", tuple(candidate.shape))
print("더할 대상 F(x) :", tuple(fx_projection.shape))
print("원소별 덧셈 가능?", tuple(candidate.shape) == tuple(fx_projection.shape))
# 크기가 다르면 실행 오류를 만들지 않고 차이를 관찰합니다.

## 3. ResNet18의 Stage와 Shortcut 구조 확인

**연계 이론:** 이론 2 · 2.1, 3.2, 4.1, 4.3

사전학습 ResNet18에서 각 Stage의 BasicBlock 개수와 `layer1[0]`·`layer2[0]`의 Shortcut 구조를 확인합니다.

In [ ]:
weights = ResNet18_Weights.DEFAULT
LOCAL_WEIGHTS_FILE = Path("weights/resnet18-f37072fd.pth")  # 공식 ResNet18 가중치의 로컬 파일명

if LOCAL_WEIGHTS_FILE.is_file():
    model = resnet18(weights=None)
    try:
        state_dict = torch.load(LOCAL_WEIGHTS_FILE, map_location="cpu", weights_only=True)
    except TypeError:  # 구형 PyTorch에서 weights_only 인자를 지원하지 않는 경우
        state_dict = torch.load(LOCAL_WEIGHTS_FILE, map_location="cpu")
    model.load_state_dict(state_dict, strict=True)
    weights_source = f"로컬 사전학습 가중치: {LOCAL_WEIGHTS_FILE.resolve()}"
else:
    try:
        model = resnet18(weights=weights, progress=True)
        weights_source = "TorchVision 공식 사전학습 가중치(캐시 또는 다운로드)"
    except Exception as exc:
        raise RuntimeError(
            "사전학습 ResNet18 가중치를 찾을 수 없습니다. "
            "인터넷 사용이 가능하면 다시 실행하거나, 공식 파일 "
            "https://download.pytorch.org/models/resnet18-f37072fd.pth 를 "
            "Notebook 작업 폴더에 저장한 뒤 이 셀을 다시 실행하세요. "
            "무작위 가중치로 대체하지 않습니다."
        ) from exc

model = model.to(DEVICE).eval()
preprocess = weights.transforms()  # 해당 사전학습 가중치에 권장되는 전처리
print("가중치:", weights_source)
print("모델   : ResNet18 / eval mode")
print("실행 장치:", DEVICE)

In [ ]:
# 실제 ResNet18 BasicBlock을 확인합니다.
print("ResNet18의 Stage별 블록 개수")
for name in ["layer1", "layer2", "layer3", "layer4"]:
    stage = getattr(model, name)
    print(f"  {name}: {len(stage)}개 BasicBlock")

print("\nlayer1[0]의 Shortcut:", "Identity" if model.layer1[0].downsample is None else "Projection")
print("layer2[0]의 Shortcut:", "Identity" if model.layer2[0].downsample is None else "Projection")
print("\nlayer2[0]의 Projection 구조:")
print(model.layer2[0].downsample)

assert model.layer1[0].downsample is None
assert model.layer2[0].downsample is not None

## 4. 입력 이미지와 모델 전처리

**연계 이론:** 이론 2 · 4.2, 4.4

이미지를 불러와 사전학습 가중치에 맞는 전처리를 적용하고, 입력 `(1, 3, 224, 224)`와 출력 `(1, 1000)`의 Shape을 확인합니다.

In [ ]:
IMAGE_PATH = ""  # 🔧 예: "my_image.jpg"; 비워두면 내장 예제 이미지 사용

if IMAGE_PATH.strip():
    img_path = Path(IMAGE_PATH)
    if not img_path.is_file():
        raise FileNotFoundError(f"이미지 파일을 찾지 못했습니다: {img_path.resolve()}")
    image = Image.open(img_path).convert("RGB")
    image_source = str(img_path)
else:
    try:
        from matplotlib import cbook
        sample_path = Path(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))
        image = Image.open(sample_path).convert("RGB")
        image_source = "Matplotlib 내장 예제 사진"
    except Exception:
        from PIL import ImageDraw
        image = Image.new("RGB", (360, 280), (199, 226, 245))
        draw = ImageDraw.Draw(image)
        draw.ellipse((255, 25, 325, 95), fill=(255, 211, 79))
        draw.rectangle((40, 105, 225, 240), fill=(230, 170, 114))
        draw.polygon([(27, 105), (132, 40), (240, 105)], fill=(166, 82, 75))
        for start in (75, 145):
            draw.rectangle((start, 145, start + 40, 185), fill=(72, 124, 176))
        image_source = "Pillow 생성 데모 이미지(사진 없음)"

print("이미지 출처:", image_source)
print("원본 크기  :", image.size)
plt.figure(figsize=(5, 4))
plt.imshow(image)
plt.axis("off")
plt.title("Input image")
plt.show()

In [ ]:
# 가중치와 짝을 이루는 권장 전처리 사용
input_tensor = preprocess(image).unsqueeze(0).to(DEVICE)
print("모델 입력 Tensor Shape:", tuple(input_tensor.shape))
assert tuple(input_tensor.shape) == (1, 3, 224, 224)

with torch.inference_mode():
    logits = model(input_tensor)
print("모델 출력 Logit Shape:", tuple(logits.shape))
assert tuple(logits.shape) == (1, 1000)

## 5. 실제 BasicBlock에서 두 경로의 결과 확인

**연계 이론:** 이론 2 · 2.1~2.2, 3.2

실제 `BasicBlock`의 변환 경로와 Shortcut 경로를 분리해 계산한 결과가 블록의 출력과 같은지 확인합니다.

In [ ]:
def inspect_basicblock(block: nn.Module, x: torch.Tensor, name: str):
    """실제 torchvision ResNet18 BasicBlock을 변환 경로와 Shortcut으로 분리해 검사."""
    with torch.inference_mode():
        fx = block.conv1(x)
        fx = block.bn1(fx)
        fx = block.relu(fx)
        fx = block.conv2(fx)
        fx = block.bn2(fx)

        shortcut = x if block.downsample is None else block.downsample(x)
        manual_result = block.relu(fx + shortcut)
        actual_result = block(x)
        same = torch.allclose(manual_result, actual_result, rtol=1e-4, atol=1e-5)

    print(f"[{name}]")
    print("입력 x        :", tuple(x.shape))
    print("변환 결과 F(x):", tuple(fx.shape))
    print("Shortcut     :", tuple(shortcut.shape))
    print("BasicBlock 출력:", tuple(actual_result.shape))
    print("수동 계산과 일치:", same)
    assert same, "BasicBlock의 수동 계산이 실제 출력과 다릅니다."
    return actual_result

with torch.inference_mode():
    stem = model.maxpool(model.relu(model.bn1(model.conv1(input_tensor))))
    layer1_input = stem
    layer2_input = model.layer1(layer1_input)

_ = inspect_basicblock(model.layer1[0], layer1_input, "layer1[0] — Identity Shortcut")
print()
_ = inspect_basicblock(model.layer2[0], layer2_input, "layer2[0] — Projection Shortcut")

## 6. ResNet18의 계층별 Feature Map 관찰

**연계 이론:** 이론 2 · 4.2, 4.4

`conv1`, `layer1`~`layer4`의 Tensor Shape과 Feature Map을 비교합니다. 마지막의 `TARGET_LAYER`를 바꿔 다른 Stage도 관찰합니다. 시각화 밝기만으로 활성화 크기나 객체의 의미를 단정하지 않습니다.

In [ ]:
features = {}
hooks = []

for stage_name in ["conv1", "layer1", "layer2", "layer3", "layer4"]:
    stage = getattr(model, stage_name)
    # 루프 변수명을 기본 인자로 고정하여 각 Hook의 저장 키가 달라지도록 합니다.
    hook = stage.register_forward_hook(
        lambda module, inputs, output, key=stage_name: features.__setitem__(
            key, output.detach().cpu()
        )
    )
    hooks.append(hook)

try:
    with torch.inference_mode():
        _ = model(input_tensor)
finally:
    for hook in hooks:
        hook.remove()  # Notebook 재실행 때 Hook이 중복되지 않도록 정리

print("Stage        N    C    H    W")
for name, tensor in features.items():
    n, c, h, w = tensor.shape
    print(f"{name:<9} {n:>4} {c:>4} {h:>4} {w:>4}")

expected_shapes = {
    "conv1": (1, 64, 112, 112),
    "layer1": (1, 64, 56, 56),
    "layer2": (1, 128, 28, 28),
    "layer3": (1, 256, 14, 14),
    "layer4": (1, 512, 7, 7),
}
for name, expected in expected_shapes.items():
    assert tuple(features[name].shape) == expected

In [ ]:
def view_map(channel_map: torch.Tensor) -> np.ndarray:
    """특징맵을 채널별로 [0, 1]에 정규화하여 시각화. 모델 계산값은 그대로 유지."""
    arr = channel_map.float().numpy()
    min_v, max_v = float(arr.min()), float(arr.max())
    if max_v - min_v < 1e-12:
        return np.zeros_like(arr)
    return (arr - min_v) / (max_v - min_v)


def active_channel_indices(fmap: torch.Tensor, count: int = 6):
    """공간적 변화(표준편차)가 큰 채널 인덱스를 시각화용으로 선택."""
    per_channel = fmap[0].float().flatten(start_dim=1)
    std = per_channel.std(dim=1)
    return torch.argsort(std, descending=True)[:count].tolist()

stage_names = ["conv1", "layer1", "layer2", "layer3", "layer4"]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5), constrained_layout=True)
for ax, name in zip(axes, stage_names):
    fmap = features[name]
    channel_idx = active_channel_indices(fmap, count=1)[0]
    ax.imshow(view_map(fmap[0, channel_idx]), cmap="gray", interpolation="nearest")
    ax.set_title(f"{name} / ch {channel_idx}\n{fmap.shape[1]}×{fmap.shape[2]}×{fmap.shape[3]}")
    ax.axis("off")
plt.show()
print("주의: 채널별로 시각화 범위를 별도 조정했으므로 밝기만으로 계층 간 활성화 크기를 비교하지 않습니다.")

In [ ]:
# 얕은 Stage와 깊은 Stage에서 여러 채널을 관찰합니다.
compare_stages = ["layer1", "layer4"]
num_channels = 6
fig, axes = plt.subplots(2, num_channels, figsize=(16, 5.8), constrained_layout=True)
for row, name in enumerate(compare_stages):
    fmap = features[name]
    for col, ch_idx in enumerate(active_channel_indices(fmap, count=num_channels)):
        ax = axes[row, col]
        ax.imshow(view_map(fmap[0, ch_idx]), cmap="gray", interpolation="nearest")
        ax.set_title(f"{name} / ch {ch_idx}", fontsize=10)
        ax.axis("off")
plt.show()

In [ ]:
TARGET_LAYER = "layer2"  # 🔧 "layer1", "layer2", "layer3", "layer4" 중 선택
if TARGET_LAYER not in ("layer1", "layer2", "layer3", "layer4"):
    raise ValueError("TARGET_LAYER는 layer1, layer2, layer3, layer4 중 하나여야 합니다.")

selected = features[TARGET_LAYER]
channels = active_channel_indices(selected, count=4)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), constrained_layout=True)
for ax, ch_idx in zip(axes, channels):
    ax.imshow(view_map(selected[0, ch_idx]), cmap="gray", interpolation="nearest")
    ax.set_title(f"channel {ch_idx}")
    ax.axis("off")
plt.suptitle(f"{TARGET_LAYER}  |  Shape={tuple(selected.shape)}")
plt.show()

## 7. 이번 실습 간단하게 요약

**첫째,** Residual Block은 변환 결과 `F(x)`에 입력 `x`를 더하고, BasicBlock에서는 그 결과에 ReLU를 적용합니다.

**둘째,** Identity Shortcut은 입력을 그대로 전달하고, Projection Shortcut은 두 경로의 Shape을 맞추기 위해 변환합니다.

**셋째,** ResNet18은 Stage가 깊어지면서 Feature Map의 공간 크기는 줄고 채널 수는 증가합니다. Feature Map 시각화는 내부 표현을 관찰하는 방법이며, 채널의 의미를 단정하는 근거는 아닙니다.

